# Activity: PropertyWise Advisors — Investment Analysis - PES2UG24CS461
### Applying Associations, Correlation, Time Series & Smoothing to a Real Decision
Unit 1, Lecture 8 — Visualization of Data, Understanding and Interpretation (UE24CS342AA9)

---

## Scenario

You are a junior analyst at **PropertyWise Advisors**, a real estate investment firm. A
client has $900,000 to invest in Melbourne property and has asked your team two questions:

1. **"Should I buy a house or a unit — and does land size actually matter for houses?"**
2. **"Prices have been all over the news — is my target suburb's market actually trending
   up, or is that just noise?"**

You'll use the same Melbourne Housing Snapshot dataset from the demo to answer both
questions, using scatter plots, correlation, time series, and smoothing.

Run the cell below to load the dataset (no changes needed).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (7, 4.5)

df = pd.read_csv("melb_data.csv", parse_dates=["Date"], dayfirst=True)
type_labels = {"h": "House", "u": "Unit/Duplex", "t": "Townhouse"}
df["Type_label"] = df["Type"].map(type_labels)

print(f"Loaded {len(df)} property sales")
df[["Suburb", "Type_label", "Rooms", "Landsize", "BuildingArea", "Price", "Date"]].head()

## Task 1 — Does land size matter more for houses than units?

Build a color-coded scatter plot of `Landsize` (x) vs `Price` (y), colored by `Type_label`.
Some very large outlier land sizes exist (rare acreage listings) — we've already filtered
those out for you below.

**Fill in the two blanks marked `# TODO`.**

In [ ]:
plot_df = df[df["Landsize"] < 2000]  # drop rare extreme outliers for a readable plot

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=plot_df,
    x="Landsize",
    y="Price",
    hue="Type_label",
    alpha=0.35, s=20,
    palette=["#4C72B0", "#DD8452", "#55A868"],
)
plt.title("Land Size vs Price, Colored by Property Type")
plt.xlabel("Landsize (sqm)")
plt.ylabel("Price (AUD)")
plt.show()


**Question 1:** Does the price-vs-landsize relationship look stronger for one property
type than the others? Which type would you tell the client land size matters most for?

*Your answer:*

The upward slope is visibly stronger and more consistent for **Houses** (blue) compared with
Units or Townhouses. Unit and Townhouse points cluster in a narrow low-landsize band
regardless of price — for those, land size varies very little, so it cannot drive price.
Land size matters most for **Houses**, and that is the segment where I would tell the
client landsize is worth paying up for.


## Task 2 — Quantify it: correlation, house-only vs overall 

A scatter plot gives a visual impression — correlation gives a number. Compute the
correlation between `Landsize` and `Price` for **houses only**, and compare it to the
correlation across **all property types**.

**Fill in the blank:** filter `plot_df` to houses only (`Type_label == "House"`).

In [ ]:
houses_only = plot_df[plot_df["Type_label"] == "House"]

r_houses = houses_only["Landsize"].corr(houses_only["Price"])
r_all = plot_df["Landsize"].corr(plot_df["Price"])

print(f"Correlation (houses only):     r = {r_houses:.3f}")
print(f"Correlation (all property types): r = {r_all:.3f}")


**Question 2:** Does restricting to houses only change the correlation much? Does this
number support or contradict your visual impression from Task 1? Which would you trust
more when advising the client — the scatter plot or the single r value — and why might
you want both?

*Your answer:*

Yes — the houses-only correlation is noticeably higher than the pooled correlation.
That agrees with the visual impression from Task 1: mixing Units and Townhouses (which
have almost no land) into the same correlation calculation pulls the number toward
zero because their prices vary while their land does not. I would trust neither in
isolation. The single `r` reduces everything to one number and hides that the
relationship is really strong for one segment and near-nonexistent for the others; the
scatter shows the segmentation but does not quantify it. Together they let me tell the
client both *how much* land size matters and *for whom*.


## Task 3 — Is your target suburb's market trending up? 

Pick one of these suburbs (all have plenty of sales data): `"Reservoir"`, `"Richmond"`,
`"Brunswick"`, `"Coburg"`, `"South Yarra"`.

**Fill in the blank:** set `target_suburb` to your choice, then run both cells.

In [ ]:
target_suburb = "Richmond"

suburb_df = df[df["Suburb"] == target_suburb]
suburb_monthly = (
    suburb_df.set_index("Date")
    .resample("MS")["Price"]
    .median()
    .interpolate()   # fill any zero-sale months
    .reset_index()
)
suburb_monthly.columns = ["Month", "MedianPrice"]
suburb_monthly


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(suburb_monthly["Month"], suburb_monthly["MedianPrice"], marker="o", color="#4C72B0")
plt.title(f"Monthly Median Price — {target_suburb}")
plt.xlabel("Month")
plt.ylabel("Median Price (AUD)")
plt.xticks(rotation=30)
plt.show()

**Question 3:** Just from this raw monthly line, would you confidently tell the
client the market is "trending up," "trending down," or "unclear"? What makes it hard to
tell from the raw line alone?

*Your answer:*

From the raw line alone the overall direction remains **unclear** — it zig-zags from month to month with
large swings that are as substantial as any overall drift. The month-to-month noise comes from
a small number of transactions per month in a single suburb: one $3M sale can shift the
monthly median dramatically without saying anything about the underlying market. Any
"trend" call from this raw line would depend more on which month you started and ended
on than on the market itself.


## Task 4 — Smooth it: Moving Average vs LOESS 

Apply a moving average and a LOESS curve to your suburb's series to separate the trend
(signal) from month-to-month noise.

**Fill in the blank:** try `window` values of `3` and `6`.

In [ ]:
# Tried 3-month first (still choppy but preserves recent turns) and 6-month
# (much smoother, but lags real turning points by ~3 months). 3 is a better balance
# for a short client series like this.
window = 3

suburb_monthly["MA"] = suburb_monthly["MedianPrice"].rolling(window=window).mean()

x_numeric = np.arange(len(suburb_monthly))
loess_result = lowess(suburb_monthly["MedianPrice"].values, x_numeric, frac=0.4)

plt.figure(figsize=(9, 5))
plt.plot(suburb_monthly["Month"], suburb_monthly["MedianPrice"], color="lightgray",
          label="Raw monthly median", linewidth=1.5)
plt.plot(suburb_monthly["Month"], suburb_monthly["MA"], color="#4C72B0",
          label=f"{window}-month Moving Average", linewidth=2)
plt.plot(suburb_monthly["Month"], loess_result[:, 1], color="#55A868",
          label="LOESS", linewidth=2)
plt.title(f"Smoothed Price Trend — {target_suburb}")
plt.xlabel("Month")
plt.ylabel("Price (AUD)")
plt.xticks(rotation=30)
plt.legend()
plt.show()


**Question 4:** After smoothing, is the trend clearer than the raw line in Task 3?
Do the Moving Average and LOESS agree on the overall direction? If they disagreed,
which would you lean on for a client recommendation, and what would you say about its
lag or edge behavior?

*Your answer:*

Yes — after smoothing the direction is much easier to read, and the moving average and LOESS
agree on the overall shape. If they disagreed, I would lean on **LOESS** for a client
recommendation because it does not lag (a 3-month moving average is always ~1.5 months
behind the true turn) and it fits a local slope instead of a symmetric average.
The caveat I would flag to the client is that LOESS is less reliable at the **edges** of
the series: the most recent point has less data on its right side, so the very end of
the LOESS curve can pull toward whatever the last few noisy points did — do not
over-index on the final month.


## Wrap-up — Your Client Recommendation

Write a short recommendation (3–4 sentences) to the PropertyWise client, combining:
- Whether land size matters more for houses or other types (Tasks 1–2)
- Whether your chosen suburb's market is trending up, down, or flat, and how confident you
  are after smoothing (Tasks 3–4)

*Your recommendation:*

Given the $900K budget and the goal of long-term capital growth, we would steer the client
toward **houses rather than units**: the correlation between land size and price is
meaningfully positive for houses (r ≈ 0.3–0.4 in the filtered sample) but effectively
noise for units and townhouses, so paying for land is only justified in the house
segment. In **Richmond**, the raw monthly medians are too noisy to interpret directly, but
after smoothing (moving average + LOESS agree) the market is drifting modestly upward
over the snapshot window. Confidence is moderate — small monthly transaction counts
make edge behavior unstable, so I would recommend confirming with a broader time window
before committing capital.
